In [ ]:
import pandas as pd
import numpy as np
import os
import re

In [ ]:
data_info = pd.read_csv('./data_info.csv', index_col='df_name').sort_values('row_number')
with open('./feature_order_permutation.txt', 'r') as f:
    sorted_order_dict = eval(f.read())
    
data_info

,target_name,task_type,numeric_cols_indxs,row_number,df_test_len,col_number
df_name,,,,,,
iris,target,classification,"[0, 1, 2, 3]",120,30,5
wine,target,classification,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",142,36,14
560_bodyfat,target,regression,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]",201,51,15
diabetes,target,regression,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",353,89,11
hack_processed_with_rf_reg,Netpay,regression,NaN,411,103,24
hack_processed_with_rf_class,Lithology,classification,NaN,411,103,24
breast_cancer,target,classification,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",455,114,31
travel,Target,classification,"[0, 3]",763,191,7
credit_g,target,classification,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",800,200,21


# txt --> csv

In [ ]:
N_experiments = 5
N_gen = 500

dfs = data_info.index
strategies = [
    'standard', 
    'batch_training', 
    'batch_anon_training', 
    '30_70_batch',
    '50_50_batch',
    '70_30_batch',
    '30_70_batch_curated',
    '50_50_batch_curated',
    '70_30_batch_curated',
    ]


for STRATEGY in strategies:
    llama_txt_folder = f'./generated_texts_iter/{N_gen}/{STRATEGY}/PAFT'
    llama_csv_folder = f'./generated_data_iter/{N_gen}/{STRATEGY}/PAFT'
    
    for df_name in dfs:
        for exp in range(N_experiments):
            df_exp_txt_path = f'{llama_txt_folder}/{df_name}/X_syn_{exp}.txt'
            df_exp_csv_path = f'{llama_csv_folder}/{df_name}/X_syn_{exp}.csv'

            if not os.path.exists(df_exp_txt_path):
                print(f'{df_exp_txt_path}: was not generated')
                continue
            
            if os.path.exists(df_exp_csv_path):
                print(f'{df_exp_csv_path}: already processed')
                continue
        
            with open(df_exp_txt_path, 'r', encoding="utf8") as f:
                text = f.read().splitlines()

            if 'batch' in STRATEGY:
                # Skipping empty lines and remove "Sample i: "
                text = [re.sub("^Sample \d+: ", "", t) for t in text if t]
            
            gt_N_cols = data_info.loc[df_name, 'col_number']

            cols_all, vals_all = [], []

            for i, row in enumerate(text):
                enc = row.split(', ') # [col_1 is val_1, ..., col_n is val_n]

                cols, vals = [], []
                for e in enc:
                    col_val_pair = e.split(' is ')
                    col, val = col_val_pair
                    
                    cols.append(col.lstrip().rstrip())
                    vals.append(val.replace(',', ''))
                
                cols_all.append(cols)
                vals_all.append(vals)
    
            cols_all = np.array(cols_all)
            vals_all = np.array(vals_all)

            ds = pd.DataFrame(data=vals_all, columns=cols_all[0])
            print(f'{df_name}, Experiment # {exp}', ds.shape)

            if 'anon' in STRATEGY:
                ds.columns = sorted_order_dict[df_name].split(',')

            os.makedirs(f'{llama_csv_folder}/{df_name}', exist_ok=True)
            ds.to_csv(df_exp_csv_path, index=False)